# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfWlz28aW6Hf+CgxSqQAJBZOS7TicMPXsSHb8nmxrbCX3zlVUKJBokrCwBQsl2uX5W/N9ftmcpRtobBTt65l6KlvC0n26+/TZ+3TDNM1XSZFkM2MFf6zcNo5+MSIvDD3j5NSwfivjXenFJ6dHx8YD4zja0vtC3BVl5uHls1DEvsic0ej3PDF8YWRiWWZ5ks9GhvG9cZn4SW6E8D9KfIF//yqF78XG0svWHr4TsQFtvbj43fDwYis+GFYqslUQGq+e/v3lqzfGA4BkGE/PL9/YY8PLAyNOjI23M0QO5TyEUGTCKGOPhmDsEEwerMsAXgiHunEODaUCemXkwkihi9h6LNt++/SVYS295UZg/1NvHcQAFS7Pg7i8s41Q7GCMCfUCR+Bly02w5a6nXuaFMK6ZIYFm0BE5VqPwMh+6ItYlVM+5J2ehUWTewnsPr0WUJlDd2HoI6oEvtg/yTWRYfpAvE3wEPbPlABhF1PukEGthLJMYm1p6sQ94/Fc1EYb1K5T7gapCmQwwI2Lquuoq1dRRZKxFDIhcBklcz0aNt0zkZVjQXEHz6xIHRQUR44w27qoBo4GJyTeIu1DNRo2292VeiBwxlYoiyLBIFOSRLChyI4jzwovhn0ickWmaowARVGCHgaYy6KuzKoHwoChM0HKl3q+X1VWYLNQ19iQMqtsgUVfv8yRW10murjKhrvJNWQRhdberiuTlArC/FHn1pNhkwvODeF09CKIKzocgBSoWI3Ufl1G6w57H6WiVJZFx8fLckO9eRt5ajPnPmzQfcQHHqDGwCtaj0W9nb8+MOfTaSb1i4/hBFnuRsNS9t8jxr+W62LDr2vYIptI9f/Pr/4NaVWed82R5Y9mjP16e/e0dvLBMaC0uzLFhLrzlDf4NxYrus2C9KUx7NBp9Y7wDQs4CoLYZdMtDosmNdQB8BLNqJMbKC3McXGKkwOJAEkUZLYAfRAh0k20DP8lGND4H+Np9+erpizP34uXfz86xC48fupPJBP+P8O27l6c4zuPJwyfQtC9WRu6thEvNWkG0tmdEVkAjf3ghdAmYPk4yEFvBB4/okjuIM4ZvrcKLvDgZI2PC7xUWLRLbQRpDOMEKqgOJ5EyBS2pDzoZDv2WD+JN5AfABtFuKsyxLMssEnpYyAUcPyNB6sKX+AQax6u3Y2MCwALiTBx+EavvW+Nk4OTaQe+hqX1vnFWSBMioCeQisCcIN5Cq0akVBHEQJgkvvbNksNfE9AP/F6J+AL2hwnYHgEXUDkXdn4fBsaETNYA0WRizHvUzSnWXrL5xiA4QSe0FoWarmuIJhq2k4f/r613+8eVc1iDVRzvLMxUDEb188Q5KFP09Nu7/teCuywpJFqEQmQKDE+FoS2jIUXlymblIWaVnk1o0Q6fzJBGgniN1VJoS7Xsynk5oCn4GQ9UjCaZIS0GRsA/EehWZA2qpCneSVwjNIzldEWGS7utN+Dn3OgfWFb/lIsIaPg0T55uCviuXfJ0FssXhw3vx+6Z6+fAs4+N60bUSSKhXkICos364RjyPBJkjWgSDJb9wyR+5qwrIdKgiq//vvTyZV7dsNyBfsJBCBYYHmsfwcZx6RhXRMlX7WUaZNCI0P2vahc0lqTezGG9mjLAKlLix/bARrYG3hCiTHfH6ZlaJZQRumuAtymDIf9J/pgPg1W63iD5TNRJRshV6sUerLUSPuliItjDP6A+q0bj31QGmMRssQ/hpnMZgYgl8iybkuMG3hulYuwtUYNfxq/jqJQR+EyXqeZkFcaAPBQk4KHcRyiGzZLV8UYllovEUFAQIUhd/Nx4CCBb7AZppv8o2XiuE3brTd87LYW3HgderBAIffuDdBjOQi2aR6+T5ZIJN8NL1lgeaYOTOAS00Ru8sk9PDuU7OCC4Yf8OQtVHoOuqpu6xtQ22iLAdsKP0AOtfACzJNkG4Q2WU4g/UC3gzpbgjlcSA5uwhfA0tgfUi/Q/hMH+yNtZbg/eUQPFmyowYPHE6fTRwDiIi1rWr6fxWWXHbRmNAJuSBG9a06Z+l4hLCwPVOH5VpIC3zZbtTUJMUzMFUE32mBTEBDwqU1qnr/r4JxeSVS4aZKEUGK5ci7JRrmA+7M7cCTAMbFAs7i3SXYjgPtPQI0xZVyZ8pl5XXcZ0BF5NzB1Wd5BGMkGN7mREqRmPhHnYFISIjQuA6n8LkDbZZn4wTpB0xforEQLQ1rQmk+z8yp/ZgwvozREWoA3FTivyIIFDAfso1Kg70A0FaMTALY3OkWgPUAKLAOPDONc5IDySjcokcmz5frBEsRF/cbJRQFj8UD7WKbkRFCFyFD23lLIk/cUrHkQlas5VAy5EQrs48ahqoovoTrRyFA5wCm18Hn8NdiqonmA+TmcNgSPyB/7N9gikECa3IPt4jjolAAF5ztrAW91fjFBwedUsqVZr5rFrv8ZxhqxeDz6J36YMyp2g7lG4ePeCvQscqnywCVM8jkMX4DxHRcPtLCDSXPceRxtYSoa7HoOyriowg2Vk44uPXn+xHG+eO9lYRV2AK+1wWPkdW3KNejn9cpbCndTLpQPloOr7dIsA/9ldRWwhFAPXV0Py2C03zK035owLNuhgXeNFJjyjN65Adl9A8Uq2GJLpRxkJBQcA2Xr/v4AHc6LzFo55CmS7CdQK25uS8/z60FtgH6s6NE0YGlY5muSl2kJfkIoBAm0Kr7y2/MZTGnLhpNW+GRUswXQSuZb1LFmMzEge9J4chuAviRthsXRa11AE4tytRJgO63nExt7u+rihM1YVAj9+FpAU6zArMcPjZ9/BofU7i0oHcjFMNrlCOPeAjHOB1rRC7vGAJpEGE5w8Jdm1tFo+zl6+phGKu6a/SiSwkP9mpeRJe5ARaZWBoY1zbDdtRetlXnBHCNjWjJMB8wyMz4yMGn1zpzp6pPx4hm+NrtDW5kftREYR0YxcyarT7mmRSReCKqmlFk9pUEqNAkxRn9+bECFRaIRBTPt7sQHJ5VtTfRWJddWIuM0uHweJrevvAIEQ7y+ANBhoBmc2Bbg6L7yDraGGqsAVzkG76xFx7KXqyQECTyn/pa5cDF8AVIs77owQDo0oOaUfWM8B1dh88cpoNwXaIOsgiUFSDk2+x/TyR05mZmXBjI2OSh+1PAcEXugG9wVgt76kRUtXS9cJ3MzWrZ8IMnul7uU4wBdgL2tDLbUZZsDBEpXsBgVWtB2AtcsTeIAGuoRKQfC/2zYkmBxlDW9kj3NhHoL7gHYVTTL8o5MKP0BTbj+4C80LIsdeX1NrfYrMqBubDrGJdY2rEdog8PzH6p+2xiiAZMVo1wUjIWugUuokRZFZcF6MnIvpLBtFZw2fgUUYYiZAztG4UWLAApQuF5ZtwC8AiVj3Bx2g95MuDvKW5K9+JXCX35DyapQaQIqupZ2eOdg+BFsl9xZlr4HgqqIytABgk9u3WJ1cgz8iUjbUymOP7f4AmyLDTh2N+3SDbHC/rLs+TOAsM4SQNxbiiRkOjPr89lvoTVKzCuzC/rtJrG7Tkuw2WDyUKHU1HK/XoC2Nde+t+WG698ZRRNW3TQFeRpO/jBw4iSiWaAqQ189klTL5EdSy7Mdx2nJnV6p0gowsBOkaYh+s3FQFQ/9mBtZ3T/yg+Joe3w0OSIEgADohp2+QL4wCu4TXD0k0qDLNo01Z+eL5oZnBVxX5taeeWmEhg6cgD6Edn08NZxoSyMZJjuKH33B6ACwpL2tlwUUl80LL/8C4pMRrIOGD/7J51NgLxFG269LhTVC9lFia1qsBsmBdBqYGLs7a184Z83Z2keb90/b/+yk9U1Zl87/idlqzNMYXasyRyWbJ6CAp7R8OjB7HFqtpoRve+ejoe0KcddnQl9g9Y7hTD0uKRZrXWm4oMZ0KXrdUisC8ANusw0WTH8987rjRefoj9O6Xr4Hb6s+oqLeG9bH76j177Az1SVB5A59xxbNd596KG2v3Tv4ohPo7kfofs+iQ3T9IrfhevT7qveb9hoVmTx1EkOzvV2itcOvioHeQdqj/eDVegH0t9+pB7f55mu5I4co8/0LHbrpVvEn2n5dHq1XCXIBTESBT3Atgw/sJKMRCQ/1gHingSszWa3QWzGvZ/d0VPlwZOi7y7R0ZdWWM4dcc4/dI9OFXj5t4suwliDBMlzI2oGfgg5GnTlimw1xhhjps5S7gbaUVoMb8n/cVlf6A7s3/Jbq00AStHqim6MDLrBTJJYJKDPt/rWQhkFHb269LCpTqzca80ouM4SgCjkUc29YRdV2I4HOd+Woyma6ix2/xx46cJ6PSTlxADY6eHWc0TSjJY0gZDf0RmQxTDnO1K+/nz41dkAAwboeDQVfw3WSBUXE4ddlefr6NWduAYw0g15nKjvIw+QnD+O2mPoVJsvDvEUZrGk4Bg2jRCcdKiyVnnKs2BUkLzPIXW/rwfCAIq0WMbCzf6/r1Q349qX3nGbebZPtI4DH2Q2xuJVpCWPwpafHYwN+2XA9GRvyn213VtArsA7+soKoVcQRYRikubCs6Y+Px8ZDAHNyAhfHk4lN8b9wbsH12Jj+iL8ewa/jR4/aDQHVAnPG6xDh/ATVpz8hoClC/LEG9BgeYp7E9HEvGHQ5GesvKN0M45W+2AZLMTdxHkwb/P0YDEwQbiD6W4FWCnpy9SBeiQz1AkkmC9rn515ZJEsvx6UYBDc2/AJU3JxfouAqpo/t/sCYRRku8yAaY5KWW7eQFyLN5zAbyRLTEdxMgM1Von6YT4+fDNiKCGIJ9sxNPlcSi2/N67HMtYNOzdcAlXJMXOonLg9FIt+0jduaUPNdvNxkIDo/CMseDRg+GMIBUxmUJrLyIaLisyL7hgIeYepgYiRRUAS+Mj8rMYO6yS0SmbBVpTVEKUle4c8nTemDGiLBdFKZakR5pDwJgVdloZ4Gl2MQ7suwhCZpEVazCyhRkn/eCZZcurjBvFJcZWXj2THO4iLAVTIjEu8TXJtZh5TFhsHepCmAVjBphVcUmRyIvkh3mDENFXBcLYMahtNvTA+6RlDDsArClcrKGvLVocnaztJasgDXZFMdyXcP6jJH26kzPToNVqsyF1kOV3kBrC18rQGeRZS9gA++sZ0clzYt+2p2MplctxKvVG9kYcy5AYlRhqGxSPwd+HWYgpoJHwgkBLPDWFRhKVPSCtgiFl7YnSAsZW3pMViX1ljztVo0SJJibj4ASinQirzEl+9emC3a46eG9cfTd5fGA4Pujacv7ZmMlQKRULrWKgA1jPoJnuTwkD2v+5VVDwHpK8HD9p4cZTOG2VY1uFQIw1RpcPmO17J7LLKdSgaLc8yCmzB+7EMKNhbIsRYMIl/CKIrctO0u7ctpcFJJdXl1pWZIoUpiv5cRCNQiC8Bm0YKvcP/21bMXTfXQTpeRfax9CbXg3Msr2Ac3k2FRCb/jjDTg32KiIZQ6mjoPAQNs77EWE1svtIZaIduvMeL726nIlprpTJdSd01FNyBRFLGTKWlYEab+I4FXlF5Rv2nfS4uHag418o7f0+1Xb6ijFvPDS8ecXVQJA2kUNqQB5fSSRmfBAsoYFAmm+zayRZmDRcTJ2/s5Wz5kzd1cOSDx7Mp8cVUSpjn1MpnI3FzRVS06r71I+Je8NyDbPYdHVl6uVsHd3HTSeI2GjQhFIeYcYKKl7Ra7U4KxtxXWauNggrjdSt2i9DL5bliygMlPWYZal+UC+2Ltkuc2j1PHyzJvZ11NMfdG/boGowzZyY1FMW+xmJaCGsTgybUardMzael/9IXmY4/O2GNCtrpQaS3stLQNARkt463PUsRmMRNaUtkxGMF7A4qKAN0czao5Ma+lHpLiUNe/GI84PvWjA54AzC4mmeVXmrpVK+lMjM4l/7USKOOAWQw2pgXTRU2coGuBc5ejagzWZYIBRZzH5Gp6bevZcb2RBuAfuNS5ZnVvgLHtSvW6TM0ApOMHK8JvEVAkIqO0JpE5ODB5pzjrFTx6S09GA+EMmdelDwYnWuXhD9fghqoK0qTnp60QpVZZdm+u9cySaV66G9EG91nxZ30sc5qSbm9w9trj/NJkniUuCvOWDt5RgRpDJuC1XQAVfGiFGrr2i5SMOW266XnTFLn4s0W/mSs42yDDrExsKsl21pDPyevJVMbFBeKlh5n1tspjoSyMoZAA891kyOPCnU8lBU4+biMHLGe/kR3zAJ+2kmYmnDTzX//ZlzTDiTPox31cSwiSSL7bZl7krhffXWN187NSdb9GJp/a9aMn86W6cmXJ7S7WnOPSkA/oDewRDE3jAMu3NuOAj2F3tFwVP6HsL9wY1PY+mtuGRj1136Q5bhxYgbXgxWB65K2iGw+oIUw3ntxFQptO5nO50YRCg5UaxIpX4JGNjZNrJwL7zTZ+Bi0w0QdWIYmrJkXdQt/46jV7q72HRTe99+9y4cnPxBIYCvgXaDFZvBe0WRL9rwx5mrcTlp6fedK7jsD9FLXIRgT0jrQqsQOdd5dzsTiJP4gssTxQXdNJI5aLmW53eWs278B0uJuOjR383U0Bxl3OGBzTFShVuNpVz3byWQ/GCAlZkloWwtwRXHA7pwQX/uo2Qc+GLGJvfCh1Oe9q+h7smukTLcD6V3/sDm3Lwbhd/hcoB7ANmKItKzeOjFsQQiAYsCrcbfiu6+pCVdAaqICt6eT4IZhZ8LuzOYoUMSH8+CGLrfwvzShOw6Bw840QhVLjeA3uXYYrRrwLcIxb/8bohY/l5r+Gt3wabMENph1um+Q9uQy8Qsu7YWOBYZQcTUZKG8QspFUS+52QCrW8VyBo+wiy9cJrssIQ57a2ElEbnc19ctQoURhJmgyhd2BgDfFZgwmwXxUXENqrckva0eA5XrzTY5m5WOcYw8oBidgDLysod1guF481NYNrGcHYoMxeAcYmWp64Bym0OwGAbSVKGHLPikarRc49C1oLORJVDG4/qM6GDn2EjpcCLn0gcGwO2rEb/N8Hua8aygkcr1b7b4RU8u7AUm2gFfGY8yIxBULwERIalAPGQlMZZuhvwMgTZ/Lwui2PsDgJ6xmJSiAWY1OCvMxnmNCGO1zBTAYbPe/kImNtYiCH+Msyx+2QiOodeSUB9OBvYArE9phEDDxAmVQ95TTsgNKwvRj0VmzXnUWHjUAlKuZGpRMs3e5DXSspi+a+HKyDoEDosNi1EcAHgEcNjKnHba+oLK7w7XVlA6PyRw7QhK2U4fQUhRV5E51UCMngAPGrmCUUu6g9FvRXAxVqQ09hjJtTXUznzw/NGrZOhfT1xHOoJxPkcCB445dLuDnQUH+ODpfIJIgaCdgzZKLGa8u28HmnVG93OoUHbesdlgNrtvEyQkKqXjnoprwTRUujRsCUvu+SJ9ksS207KCPBB88Z0w4juQMC3wXx2pWmj18CkaIF7qrq1p4qOJsuWBMuOBwBJje4AuZsT5VlCJrA3STArKjA6QIdpOPJZLKnHTfJcE2DCcVdJhvyPMO2T4G+R4TJo3QgAWPG7vVbWlEhbTrbPnpEeBR3MER4RZZORLjUHshI0rzFS5+5elPN4UCkTfasJ87Wm96yZ6hquIBhsCKHZpohMzZ4qp1VcOdyDk7eou/BCmGoprtTvhc/x/vzFSjixedh8PoWO7fHDSVWyRRSlqgENA4wfqnfz/ooQBMkko1rKQW6fl5LrLbUJES3NkpIYYeb6PIqxMlLlu1Qp9w+gUchgMPbdNE4GoyujQycsnfzZYsXvfQiR6+v28sQLfX9Sh5EcT0Up+0A/EaF06uM9gSMQYNZmabwgcDYmjejHZwYcPcwnaFaUhsNbNThKemaOJ+dwqf6t8Kww70RbWXt1whCPWV9xsza7SA4AfhsAul48H3BmKgMC1xm/HgzA6MRrYobabBSS05QiAg4nk1UjVJqS0TlmLIZRfCQeaa1HQ0UyKmFe1J44MF1tZFG5hXuTwC6to0/21a06ovMS2x1gXbMaXSrg9zTeKPl4rqdwRKLu8JK67wlbqudf1TTB+4lInRxUsu9iU6ayMopebmvXrGv2opqtjQKEhDqQfDFKdKO96Cjnths3qsnxs9z8IkYJ4++LJsrKTYia3W4idNW57tj6c3vknBbOV6Np4iq/vwuKtaT41Vtf6qXA4eSOUSUFjveCmq1ozjMVtBJOduEv6aA/N9ZF/knUmukpG9k1+xZNJEUNGhTd/NvsBY/tcd7U25GX7T4YjdXe4Y71pvC07MuwwqFZ/SruD0y7t7IuVEHGSjHRwp8NC2/dMWmVxMGqwqm8S/ztoWASw7KQOgxRbtrSlX/dEYgP7dJ9deYKs6Pbq5Z3/DRPwPHVpHiIU2E2+dv7D1btL8x/uD9BnXeUW5Yq4wOSPsBz5rzQmBbx7ENzmMEgxZ3WhthYng+WImi7S1XfrJHkUTCYsUJuYo74DVpPOZyvAXi+WL7Xp3Mh+HbxgYKtPepn0DeY945wIPrMfXv7Xyjh7U3zxtUvg5xJ3JTK1L2jdg1DZgkLXRXHgWNPO0N5f5UE6gVjdBZZ028oRrVqaMbVdubib1RJ6rcOEAuJC/t4UJb0AmLXYEO0NVsZvz0o4bBTeNsFr+M0tzCIY7pBCgXhi83Dfc0JLG/cTbizg/ALy6wgenxdW0GZmUs0ccBXkqxUgcMAcPIS+IZeU1RUL7uRlTdveHi9irQuGs5yEZYessbcnbkNRCx6gfmf02PTx42oSghPacTP6DdubkpcT0y8PEIjg2l+synzhNMcQLRJOYTKNicfCiazM1fPbBhvMwc8wNUMKppXmFXXQIkF7gc2DMmKcrkft5MhIHYYotPdFOaHPh1hh586EUL3zNQSM8MiYrYxShj4IU9SKeeScefigI/rxXefBC9oeoWe3BztZnGbJ8apc666c+sp+T0XtHSyPm/qGTDqG0PVpoc06wvrky6bhzQQ1MuS8obLso3ell2rrkoX3NJutYL8gI3FsMrLtTaB1FNrixY33Px6l6vJClAjUveyZHxXaMbk1Z2uJ4zSTNvARUeg3b6tzIoKK2TVkZMvZsgjfgMKVJ4My3SKnm3Yi5bndaoFyI+1Mt07RbWkXol4uAmYGZlvRA90Ut96l3N6ReXumfI1rm+EiVXY2rRYivHsddy10V2v+HSQCdaCiDnR20IlVExYPx0DkJ8Tkmg6lxTzJkj/alNH8psPNcuWEo2mDc87blyuFnW0W8WdvC/nevab3FWgq/21lH6wf9K6vGfSvDxn/EAflAG0m9d/lVXtRCUf/vB1LJxkSShVd3a40o08gu+sWsZySa3vLMHoYMIVZAXiT1WApGe8bVdC0J6Km9sJRE52FXBv1EiDC0L3abQqeSmolY+u4b8w8Z5ivVLILIrEx6BPBiyzd6qwyExSEWV0PK66U1mbACu5Uj7aEq7tcKz9xgrkkx5ka1IOpnf/vu30be+++1v37769p1r4ibMm/6D1Miwa52e1jqskQ71bLSOB9Xi8V95UfpB4myKKDRt2vCgVUY1ht2WGg1FqyWVI69O+bNW+AOPpmoP9IJPrHIJ/eOhUWCRwWHQelhf5Gr2mTbhltMumym81PLKJDvc/Rh/oixOW9MPZ4ochXak4DfGWwxDk6gJjelkYnzLp3Wgx4E7GfB0aUzJQlPCy4U6PBuP482CJNNOvUDaOLuS56ahD2WdXVXnpl1LZw5ZU2Y6ITGcVceIUY2HNa5cdQwArqPCq4nz0wQPtq4beIBtahVYQauKVY17e8FwGkdCFcHyxgIHZoHyconLTjnlBiVpa65M03y69eIPJLEXuFxv5CXMTUQOnY9JLyTE6dhwRl+29hpZCoS86cCWr/r4KLKfoAOYNZbjAlqXLO5krAwG/mhsWK0dOVMcKWWN0oimPW5EZUF4OHsLqOXhivEdjb8ngx+7c+sFBbT4yG5iMAVPSXRRuIrbkgtgNA6NPtsC0loYoETiuggfSWUVmDdUzHGuxgZc5/OBKQMO98CgiHtOdQXuoSwA64BVKCk1V3GrcG+CcYWfvLPYKdslrtWwVp02elUdZ3iN4eCpri3UOdsg7Jc3mFrYysHTTMATEIxnMWaIeDN19j3wLDoiuHqRyKN1zFaITrXQo2DanTvSO9cqpc5nbA1hELnxVp4c1hB9JDI1eblFQynv84DJJBriIBUppMCYLyPbRKCAJ2AVKTPGmnQ5dN3d5HgomtnVqUcnpybyj9k+CsQkP2hbh2HMgV3m3R8C91//ySdqmK11AABzaHdZ7830cyr2rfp8ydEiLRuytRLYM3OVpG8JrHxYPlUTZp4HURoQ+jHHMOOZ8Nble5Eludm3kgdykA83q05QBUuJjG4O+9KCrIWRIUxAliae3DH+8qm9d7FRZYUwsTWNijqxaZW41N68vZ57+CkQe9NKtFXf3sobXEyjbsvooL6k/PPceDzBH6at+1aQuaw90I4j7rBHTUuFLD2T5oDw4KzDhWl/taPbeHIJwfqyeh8pHNu4UJrR8HC3XVriOjtMdI47Oz38mIcP9XPDotPGVNQTvwkCphHQTmLfS6NkiExIypiESNwPSjbBx4oSZuNP0OxHXseHa2rc7D1foZsB0LP6/1Wo5MAUggHKpT72zT7bqaaXLQOYp6G5H7T4+6q3DP6G4W8xoZlcRp98YOzUM6az+rTDHR6IDNbt5yuX2qocOLSFX/rQixtpp7L+aRDJdMxWLOmhymT9TCFsXgRxofSRHqd/+fRzTx1qKAu17tNe8elj3N6jOgbwMJQBwaqBy9yrGBTcfRTH5o5cEQo+gIv8pcQ3BGk/HaoJbVKiosPjGU0S7Xc7XA0M7l05eCn4sL0djT0eh9m+faZgw2DUPDl08h41nbsf5CNpHYy6nMaREiKjZvMNf08DXT+sgcOzkeYLq68brUpBpziH6sNHGDEgCxpkBC/YNr9tREfJOBqkUzrYFnPpkrz65NQM/ECwKohGvers6DHaGWSR8+4s/LgMx5tqgMVi6DjGOG1HKu6hUq2TF/XhnbxWx+dp4oGbAfr3PHMzSlPmHhlxIHtJJ7PslnqqMhE6raeaPEZcI6V67HhLEJy7oopcD349QQ+8tQopq0wWUnG4OsKdYziSQ9x4KDyd2A4OIVyb2CkfLDO85s5Vm9qp/3XEWXSgfGo6uQKVvSXd4yYRcl2gZ6JBdLu1J2Pjzh4NGA0o8cDaBx/+R5B1dI8+eKM2eeONnmyHe7Lt9GS7pyfqHSKLyudrnUHOOSgEXEAxLjroAuRk4FFAKJ8hBcloHH+fyfAWWZDVhKXBErH6yoVyS0NPMQca4c7Xif9Vx41bDathD4cA/osSXIPuJvh2eI1DetXCbzu2kS37eXN136dloGZPfEdHA5Roj7PaNqofzO/k5QKsYKvrp8tSuD0gHfO6uhbgV5F9GQbXAvdVuL61PKni8VXcHSdGiQHilGYFZV1YUj9WNrXUkPpyJcv66hZdCH27xqrzsRg8U04yd8XY7UwM/OqcFEmv39Ayi0en9dOH/tZo94B4F7F2fjGeHRKjp9myVb4BjxOroB9p/OPlBdG1X9G1RslVbw+bqP+pyRqcMMbZmOVJ24Plb7MRHlizoQ5QKrPWcW+e/d+x8e7yfGykgfiAh2rylxF2xvmb03rO7v5/JNeCDDRe824OfsGfOFoJh7+/te9cLir7w9ww/4z/jDEWtLrr1vqslBuFZephy58t3EXYNZIXuqFVx9g1e0gPvCt7aBEOj6pKG7HOxkbPV30Agbem/fnbhVVQpLl5KOjYNRw3gEnH1L6k/UWirhAN2sF62Ujr00RQTIOS3u1v1qUZOKTxu3brd31tt/0nGpr+qRZlJfI3X1QKT3DHn2ppvrW7y+Eua74w8V2OKuBxNcSWZv+6N9cI7gYcDuje1Y2LdBTc4cU9nQd1SZ27urYBMwVuu+vA9VzqqicblgPTamK34K200qiRThl74HgPxHCDyhQGCRsUfZnjPv0wt2yGWkVl0JKgHFnS+XMZOlarzZ4beHN2P8bV+fYuuS9zOlfKKlSQcjoQnq1qSYhVPYIJteoSklerEovwELB4HICq0pQOE4bfXfXuhSi/qTrnbDH+bO784srkKzpvLi3xAWVX1/iuJQYiVQqNIGVBgZIcF8jw3DtOGHK9fBkEKgNJEtEc989quQ+4DbLLnhI/KCwRMsJ3brMA5hllsd2XI/PTE1wgiVIPzA4OmIDm1nj6Awmgsqg+39fyQXAfIzX+Ie2BT2fUmOd4/lHjMyRV5oYfZHOuHqTzD+mYiHPOeLoRu/mN0mH8Ldr5aqsekIU+Z0O95QaAOx7kG5etGgtq2gfsltlKzVYWXSVVy8XtPXKR2mzLRY3x7hG8+yTp1h4QRts+oFvbHhJdhwjW7f2CdRDiIdKuX+ptD5B69wk9rVtXZiUEmBZI86tW1LuB1SddeG67RRpM/TV4WmOOK5K+3Nl41f1eQfUFpgNEAMjvvkOrKu24kQKiMtK2DUlxALN/0UYtno+eJYnBUJpyySlocd0+1rflHMoPmrKf2fdNU91mWe3JM1ltaUsuNgqWQxhizp/VkC+apdcJ+B7amwZ+VCoSDDET+dAhw63leCxJz/4PoilYRqLYJH69c1GfOe18oAilO4h4FO5wYzYXweU3tZ1/BCmdjAYliMDH9ZuXF+7p2fPzp5dnp/wxRhT7IfgR4XxKhPehm5xIYQj5uV2gYDwdDxO8yJvtzjsCxU5qLb67fPP27JSmzsGPy2BnLct0ovQhGnby4DZTBTjYacKP9P7Zf0Y7RuL6RtQp/UEyS5fvVngyb40B3qKCd40EqUyA67yUiEQektn7vwUFnd4c4kFOujt59IvhbelIMlDVDmAUT3jGJCTpelL9Vy/PzwAlr8/w294f6/NDZNhhZlzVyz+00K/8qJk8b06tk3ChfuunGZlw8NDkGgxZRgSFAl+q3BAk6PzdrvpWIIF7ooMDRluWIcHjEoOAyKPO1AEoCOmkOT4qwEd5D0KJPPDC+BRccoERDJ7eXINRLwZRw+E6X2WTuB4e6u0Tph5hh864AI7oxfkzQ70+GN6tWEhqdvDAZxOB4EHgWcJfgIL3hwMDY8VZLe4IWKtzz5/9/QA40nOC+j8iGVzQLZ0+sykB4YdAUBP/I078K1z6qcjh4HGAN4cgnjwGEH/wBiGRM1pwsfr8zal9MDApI+SpiuZT+YX5mhVNW1uClAGiHs76A180+ErkSxEPcFUvBR9rcCpeYhoegiLXIoisMGZMjIDjeKuWUrAn1Gk6YLcqtx/eOgD6Onk8oYnqgKveDgDhHiuM/sEhRsWKEpmf6o0z0nBRJ5x9aXhLC2n1mFZqt8jjMUdD57gh1uQYq9oxUm0SoS0a7a/ELSM8C/5KJvw+Oz97fXqG3609IlVzdLTylkWS7Y4on65M6SGtAE3o8qK7+twXylfBQGTVdEeG3NERNxDg7BJ64AYwxHHiDlR4CZVNiT+4YxSadBKTxTcMlnEqX/CN3QeuCooy7tUT2sjBlat7Bqy2bfBLedcLupozHOJUJXbxI5ncxeg7qkU09XYomR1KytWwGmJ732RdgIywup0eYEQspiQa6kj9GWUJnrOq/2Ve6d3qGwnVYp5qoQ4X0bEwVKhW4uSdyIZar+rlwtqsoBOfm5/ETTP+Ios8xcS5IGcBKBcH4AO1zPWXLy/O6DlYpPrzd5enb36/vC9XAmM2cjvUolzRyS9T7XPLmLw4bx2zhR1GCzDNHO5NJ+/bfEpHUxmrzIsELV1QHVmXU1XZ6DOfYxy4x2qMyHoGvxQ9WCtrQDSsP/0fbJg5hNbre0fVSosUCvxVJ06hn1fiv99vrGtZmIZMWcjOCZjYtLnTwePWU2tqYx7yQ/o0RI+U7vFJ6fDYuOXy8HHf8iQvQk4mj6rqrq1JgcZfGBs2MXR0D9sOqlQP6lVDVjPPqF4R3Xn3LIq2IQ18xadDDFco8ez23HV7KA97s27EbqzSupsHv0X5ULAFnBfjlzlTNjYEMBQuhj/YJPmADh8b3tyuEU471xx4hTLMNbZP1sUd7jUAx91k3UHE0IjGp/iRapSkuJEVZdNkYGWFoV0dnUwmk1l3uzG/Ho1G7tnrFy9fn6kJGY1Qc4O8ckW8xu8dQMF5mgGhS/Stw2ThhYasNpLdUlA6e8Fq8Gc1vCoeITujgP03wB0viQ==", "blender_rig.py": "...", "app.py": "eNrVfdtuI8mS2Lu+Il2NsyxqiiWSkroldrMX6nsb0zNyd885hjlCbbIqSVarWFVTF0pqQYBfvIABGwfwywILG94DGDYWiwOssQ82vG/TfzI/YH+CIyIz68YiJc3MWcM9GKkuWZGZkXHPyJBhGG/DTCQz/pldiCkzXyfc86Mu80TARJrlcDOCmzBLInoWRpmYRtE5i5gbLSMW87kf8p04nwa+y+FZiP9nCU9FyBlncLUSKXzK8pBleQgg4Pp5EOXeLOCJsA3D2PGXcZRkbB5EU339KY1CfR2l+irhoRctizuhr1LhJiIrmqX5NE4iV6TFk2yRCJhYOC8e+EuxU3RMk2Y8haudnVkSLZnN1DuYzcyfq4cihNkK/WouMkc+2dk5+ebtuw9szCbGc74EjCSGxYznUZIIuvrAg9zj+jKTV8+4H8ir9yKO0giv3vEwSvHiDZ/iy7Od0/cvP7z8iLCvdxj8ewBrsJwmYsTMmKdRarHIzRIhLOZyQLvFMnGZ5Qnv2mzQZ9QEFmspAC7AWEawihH7mCfTiMAZdGkAtEOL7R8dWOxhH/5ZbNg/OOpass0LkfEg4B61O7LY4WBoscHhesP3PPZlqyGAGx4+tNjBeqsTwAC2OeirLo/W27zjl/6SWh32Gx0e9I8fQrObnZf/8uTd6dcvPzgv3r6HuRl7SHpAq3tv8vAq5+H+i95wj6cpkMaeuOTLOBAOAJ2L1NjZ2fHEjDkzPxCp6XVH1GuUeCKhVZR4cngI7b3IngdTXBT1FDil8STx5/Zselk+saeBCL1KC7kq/ucCGnVY/NPNgOd45ruRHU0/Vb4uHi+zoPIYCDERqR+F+56d4psG0NgXn3mqxxoLoAV9wxPXhyXVtzABxw9nkY2MB1SHn8+QYCbwScyzhf0p8kPTs9isy2ZRwmbMDxW+/BnTjcSln2apuf5NtwD5FcCclSBS4CThmcj7Nv5Y+9bYNbrdbnVi0N8MBRF+DgBBJtDdDNjTSy/8bGGahh1Ec8PS3YJwyBNsrJc9zZdLnlyZOGW19hwmSxgAtjYN/ilPM4GceH0jOw+AzwkhxVDMGVAp4BAlGvx8ezJiu7vXCGPSIV50kNU7ZyPrRvLm7i776S//A4zGy13fQ/Fo4GTKXsuvjC4TQSqYARdfFT3ODNUBNZeddCzW+fNOt9rJj/9DCwF2zWVTuO90b1h8iS+NCkCR/pCLQGQRqwAGasDWZuXRIkfqgc768ELeWIwoioEwrIGsfBWrjzo9gNfFvisvUzHPQZ6nThaBeFHzuGFphYxnxqsoWfIRu+4s8yDzV0BevFNDWvWFQlpnAPSVRZ2b2rDMtq5nCJ56pdF9VGjDtaxNqZP6jW4Vhh2/7DbELlu7UY3LjqrA3wPdimRUWwB6pBFXQ8hPv//v7KMvQAWlI0bDZ5snVuK0NhlmktTvNqeED4vZwFyqhLRlVkUvzZk9C2gerR9Pg2KO7R9fm52V7wkgsA77iqVZYrZAkS06XeASaJYyU4BIkS9ZHIAyJWzDnGoTXfu8mLJR4zUizLuQ7O6uWiMpb2pMTTrEBSEN4kTJmkKe2DyOAQ2mcVI2AsqDyaBQljKQBEoNypmSh7VugshzSABs7uTrb1/Ak7lIorVOaooDcH+OhLwqhErXIIl9brEVSl05pLLHM9vPxDI1u20DW3IwQpxEBL5Yic2Dey8bjBi2R5EaIgWDdmYu2m5ofqZS0CqyM1r6WokE1KRA0gz4xr5Aqnz5Y4r2KgDElqlivkmnBkCK7tWXv5XPqPWX/xXCh3suDHJvycM5GI1GXesW2jfwpwIsyyiVCgIEpS+SEH5fsWnCP6PkBOwxkL8wV6BaMG/TeS7CtolJiXvbtN5QKz2rYlKVjztnsLCIw8o8sOspB2ZxhbTgAShYB7zbNhCS6GAqoJ2FFuugy56ywcYRnRYawuUB/AS1LFECfAq+gIKjB1oDTkMFO/7yii2//AHIQCT4aWVU5XKnfujgx0BmiFPowNB0un1oIzCS++w3SoUvSZ2nPqwC4+gAad24YYm1UqnQzfpIcB76NSue6mkgW13osSqBAcqMnIDJ2eal/umv/+Z//8/fs+uLGwVI2TjG9+H3oWJq+qQLNs/7t6/JgTBO8gytc4PntHhAMEDD9GRBVyAOyXFJfE4sBi/cnHtJHguPPvgAIy3Qgq9DcKpyAHGjTKuVLy5E4iyyZWB6429AXllsIfz5Ihs/ROMd36vHF34ixq84SF6LgYMRc/V8mc7pSk0ePMSX0hNl+y+Yia6cADO1iyQETqU/S/hSAA0nIMwrUgIkUwxQyWsFIxmMBDS90d1U1IM2owcmLF3cYsAayhe2cV5gj5bLotAOK/LE81egpq4CMe7IGY+u5e+b+PKx56egj65Gs0BcPuYghcMeycyRK9ABf7xOYQaaoP7sqqe8Gt0SIfQ8wJ2bgToYuVGQL8PHcx6PBn3oaMrd83kSgZoaPRgMB/uDozbYUzLde+j45ulocAAfAqAoGT045sce549n0GlvBr5scDVKr8AWXvZyv/O0BVR13vRV6n8Wo4OH8WXn6f/5z//t3z/ZgwZPsdVT1DkmLC9i3fiQTzEqwMlWA5kY50HKWVsHUw1eDVEciWMhOk9fCxBUPEGyQLH19gTAv9TE+WRv+hSNZ/heDoB+aq5Ddz5PAhDRsNBJWvj0P+RRJqjJD8AwMyMbA4NnJoYLbPxhdtkuigywggtCQpIuCeIH9HBmxp+txtf4omyG5N5oZvwZ8cCgaENssNZIMgcij4Zn0r2cSJq4NM49GcNwwP/eQ592fN0k4U6VhMFo+fPrH9TYKiSsmSlxx51r+HnT0ai/8L1sARTW/83jdeKW1DTqNyitjcrWybMDojaILoB48iBIQdmL8DG7THopDN/nQS9L4BOQMp0GcPlZ+RGssBx+g0jrnInGQ4/4b5Tg8B+DHzj3w14WxaMDJNknnIGEmRXzz+C9yMYdMFh5eN5pEKhRp8yj6dF0NntcssFgH2GeTBM/QXkV85AiKRg8iwMUTz/95V892eMlde7sPGC9X/CPzRVToM4lQkkB4js/XUbSWwBeQ4sDxgC6hEUzHzQ9BeWKoAm4FXkoTU3QhThOVNcu90TIPQJnSkXZe1q4B3BZqIWuzZ5DQ+JtT6zg4QpIKi+lMDA7fK1Chw/AJmBLsFBgnVHpgoijABrA43ICNkOtA6/DdOljtAMxufJXEQt9aI6MS3EVgG/v/PNvn5Gq0/rIueB+ZqaZxUIgDXBZE7BARJpWlUsssGtCCygL6GjKP0UsCATZAThWGgf7CwTxF2wZ4VjRw4DWfIV2U6FXLhbAfNSVjlCk2cTAe7CSi1AFPvMAv8ZZyet6YCbM3uzbx8cWNZuBtW+hVnPHYJbDk84SDQr0pKtCqYeNO1kfTFa7P0OHrwyZULs0ECIGuIeF9YTQwaSqjiHhPojCeWK/TJIoQTsZeQzsSAQObaFnIlEZQ4kjE8m4xGS6iC4cPQ8nwuAOrG3qYySZow1EiwekJqPMuPDcW4IaLNWylMJ+CGviZjImd46hvSZoZd5RMxsMthDtFWHC0J/lWRaFthv47nkXhTvgHmgqrYq663V4IzbByQBqMZx1Lh3CkopwtaRRo6b7gH2n5ofWGukgIkkil5X4PELicEWQBzxR5j3Ql4fWZciWYgk0y6tDqtpNaAAZL4nkFDd4wAQfI6S4NPvyB6T/H/9uH4lAW0YqUPnjP44AqymbgkpF9t1gtyrZdSJjgMCOp1Hw5Y/zCGPEihXDwh4uV1wZRxTQk1iIYW3Wg33VecUt4cG4S5ygn4J4RVmJj5+yQzDHCfdoAaqOwfqIHRqLCYo7BF5GLWKxQMzgmsQ4MMtCCP3LIb0DrwDPK+FM5xbBKGPlTbTMOBjnFDqHlkLALJDBHXk5z30PmZy60tYs3EQZzwRatJ6I1iDSU3ACAGpKkWcEDwjJ+DQQOkiPIKXbCyRYXIMFzM+Fik2sAfYoFi/GH5McW8JCJ2OjkN3MTL78AWPwXYNk3TLOxoZRir0xcMipljTdihAkahsA9fDwM6+KwatShhceEtAfuAXouNGKiqpZXSCuFCt4B4Qid29s/IU2FQb8d3f3ByC7BpJkRDiHZuW2ilnIKlp25qdEFUQ8tM76SdmVsutNOXWyM42uDQTtx2bFam8Rdg17FFnNTXz1yKOb2PW//ENoV0RrIbT7dn+gxDR4T4K0AyCKdhtC5ZQgUmHZo5/+9X+tSufkqj4uOdcxYsPG9k4WyV0LNSlJoSUAcekKmOpL+oUCloO63jrVmfENCaQ4h1FidCXZOFaQ/eLGaF+Hso/qkMHN8TOHlsdc58mudHHpAwX2k4/Eobbw7Cw6F6GzEJfmQ2XhZuS5zkBCA5Jx7wPdzlPpSIfSK4UlgIfSeySdVtyQlh0x6WVWcKJUMkj9M2zlVz7J+nBd0a039BnaFRMY6hmONStc9yjwUBMFwAUmNulORr1hv6JS8aEdR7EJLS3qAsSp5OMZkQ+INGDjCm0qrY+bdfzSVHcgVoOIZ9C4sheC75ayZQkzCh3QtkAt0t7xGpBxqviFV3uqDBQdXsDbyjAvouS8yj1rJIsgErJwJA0kefizJfU2jaU6q0pyLbrJSxOXXS2/C6F9KzwMjhBVknSHH91NMr4i1G8f5Rahf+vHShGM5ZqrO4phl+pCaklkiVvBUVx/jBgi3HXZkzH4sBWlgBe3g5EKaTyNIjBRSg0F6NI0N9YXtwPTaoygqZvu7Z8pfYfyPErnhrRjE5Meo5znSSa3AI2P2KTYRltIBWl01+hWM0RBw+rRfeSrTioAT1Ugwa91QnY2OerX2VUMHnzXdhzkMse5kSJ27RMCRh8VYG3pwIEd5pr1icSJXN3is/L1DMzHIGhhWOmAAHy0JCSzF2kS9ke6MpXziwIARAkHJg3J8FCoVqMofSxDWowVN0sKEYXfErFFzEFKCRCsVj10qCKFxsKnRa7YnrIP2iWl3Wu9xG7ArwwVVSTjaDsxqaCKMnz2yJp+p60bBI7+ahTkpPONBmmCKs1jD0SDmYLL62bCG6upNxriLlJLF6gv+O4uM7c7ceBJn8LCamtCOdtoP9R7ocVHo6BbtZhVc5OQezcDcDgqrT0wAVS2CAbXFLDC0iO1TMoNRSd0UfEr1w2Epq31URmX4BiJUCYMedFjpnrmOoihTa0KgenNn7uTWCtZARjDqtPAUC7QR727dPuKZ+1NyzWvg7zHqjeCTKdR6AsKOBQ7ElessiMoLcrK0hPT32fh90clbKsKGjtK3AW4tun/87V3cF6Os7b2SVV8S2qwaD8FnQ5FEhZe4TPVYkVWZwgrY5rbUlxUfgo0wdUDUKZXz0YpUlJq2kdaCoZ0dI25n0T7D/tGkbVCj2v+dFcbhwjuM/rVHsau7c9+LJFOmUrw2Jx8js9anOrPsRKEkzOMeheZTZt5Abpso2oVJ25SdTUEYtXvdhpWglUT1xIVCgX2Mj4AyHI2Tbb56T/+G7a7+04KHUQ3Usgtca7dXdrwwhnXknrqsEFv+WAhqAFrVpnyzF2YcijSt1d23laeUS4mfbaZysmbXPHEB2sBHUodkNvg48IywxiCCF+iSzI5K1wM35KUKMJ8CYwBa0Vdd1tChz7bA1M7VA2KsOGriLZYAUeDm73rssFN1ZMFUkIDhXZ7Uz8EYgbr2Zxh5CRRlDWz0WTZ7AYkFfMf4RFSx232NeF5TD/rzIR40F4IMCxS/1m9BeCosgsKFFPsdEwp6RTIBO+6N5Rrtf4ymXQAauesNvs72Hj1fv/Tv9vYb9VlVnwnV7e2NRvNNRUurxzSs6lZCuXXPBBAOhgrlTo4BYeGJx5tnZuU7ASrLlbQJJLxCmg5S3CDsFsGY3B/keipoCXv9pQ7mfBqf/vdR0yrVPl36KBh/FZaVt21eIuG4KdSQtZRh0LeD/OScvzl/E7CF+UoTNWRU7XjcG6QQMFnZDnSo26roXenrMSa1NW5Dst5w7pHPOq1N+G1xdaW3gPHf3B4ZiciDsAANw3UUgYrUhcVIRCoyehh/0wtfgRA5fKbYpWNUIB8IEn8gmdcoRFQSsvo/ZpLuLMBV3oFSUPx8Mq8HYnFet3qvRWZk3dbV5W9iYoacTABDNl+6InLs50W2rv31r3OFl/kmR+U4RC6td0ovqrD0jcwFFpyh1Ss43TX+sCoixyhtEGuZQipJi/VK0y3tcHLBwsDKGF98PW8XE1NTXFVUQU8Tf//tG/+tCbK3cySGrT7GlpW0wLZYHUs0LCBRooSPyNv35efCW3tPK1VDoiZw0LMoCubJ64odEydFiUjxGmNEXRQIwLzv3i4WsJoZUN75SdZzgOHdrOuKhEJFO70me3mHrfhvYMGD2LFpChW5aWfOnyFByGmAdp3hFjcGuhXlHObwVRPXLhWKPJU0N4Udty9QX37/uQdKOTV0s5T4YF5hFsO/ZE9ANtxr2V7DEBBW5n3WTRGQ/P1M0yIXNqxSDAP5+Y3lFj6+vQ7lqfc46N2WOZ8MjgDo3U+6QOxNHq/ppfNXmpZsCWoYvOcUudc7i4EzAttOWHTXWHPtNoyl2sZSxg2E5dIlM6btx8+UpA90wHxeZzry1XCl/q6cumWLXALnq6L3VIZ/NS0tk5eSCArJBKUA/EF/Ojb/XW6RAFZnuUhs9IIgW993kuXPuv1wGFNrnow2DGC9j9znK8N95akSVp0fU2Lam3fEcWpoI2dJ4LAxBHIIttL+AX0JqNvYzddWWG0EBwD12GUA4+nxtZgU7oQQaA27FweI3QnyrM4z9RDDCToS/Bz4N14H0NsHlwVgPGAwkRGhC+lAL+kMxh6d0vuvJhgZRpdIDd9axndMpaJLg70EVsS6xzaWZQBuQvXRIy458AnQ3wM4oVP9s/w50EJAunByS4zGc1UXw36wwMi671r/Lh4ImnauEVpVUAauD0OXCU/uVXarHiQknXgZiaSAc4PgI1XuIELvwp+BcTDawUKLh31HFUDLvo4i8t8zHPHkrhF8Dq1uRwu8cvk3Cm2Sy5LkYTZhsX7yah33K8FoxCgVcwXrpa4EtoRqKmL0hdI+AxzlXXSi3ncpxjR++hTRNuwXsR4MMezeKlMdwx8TKoA9lnwKzC+gzJqV2Ov9sEAOgv+3bmDb6TP1tSyrGq5hJ2n7+TMMOSDeZBwB9Ke/CSV97TNNf4UTVP5nGcos0JYZAMf4uEcg7sZxqZwdxDtsNDB+eKdOrbz9dt3VQI5fqQo5HgoSeS4L0nj+FARwtG+2ncEB0we+DENJEglFeHXg8Pp4XTWx8vfoLFnGr8FLWNoWYktjqePprOZbrHTtH5Vc916MDv2Hk4r8J7L/lzVn+jz/n5/I7SPKH+1HMb2B7PpvneMlz/+/XPN/r+z2BuYDSXhHkthixYGUIC7APe0HndA9zOru45yOxFZAzBssZyMR8JRSQsYQSiovxTjKzpUtyCjcDHpgcR4OsaFqbXCPEaY6uHB0YFHIX/6jOwB6LDqoOGLuodGw61vXlbao5JcNNLUafoZztrQRyBAmv0O5CDGUI6OSZhZ129Ad7+Bh5iTdUmSskuyTerwrbpEhW4u66EbmP5xf3TWMLFpIcrowpM4Cq4weZvFMLAsHXeuYaw3HYw5BeNOCHYt5mUm0Tmw2jUmKW4ZycyQLXuUwQnth/ZhHb0D++FNZ+9pxY8miqiMJ415qFk75h7uEI3240t2XOR+Fjmew3qO58bE+euOWuv9/U59OJ0HQz709h92blRKMsxQZ5gOoNM0CsC32QzY3QSu8/QaaeOGPZk+vSY6JAV1nd9grvCTPZylxgJKEXTPQMzIjAUtWQrLFDqhRk9ZvySrGiGSWRrwGzLYokAfWPmTIncdsTTIDYjVTdcaqgzrm7aEb+hDhfBTwmSBo46SxfKQIOIUjVkCi+0IF+uYBmXlEK2jPfGEri4H406/wy6HQK2/A+q+gvtrZMO+PcDbYe1WM4Kay+GhftTzeLrgScKvxp0DdoAkTj2i6YaB9JbEcomNpz/99d+wD3QCwouaJ6VoiVVeOWWYA+qkuPy+9FhoQ7K2sqoHKeqhh7/6vUzok6ecNEp2Nh8pqCZOD/vD4XC6zhT1BZWjai59Mzm6hcQ0FR4BYHqA+cyUysyKowCPG7n423Kua2cfmqcaYOKu6E1FdoGJ33S44SLh8Qh/0KkGOkhwjYt2s9ZJC4a1zYE6GRw/Zd1Qxqx80nQFQQTsNQFvdgIx8Ta+kLe/U8tWNWKqg1vNabfxWXQJ9Mz6DIiZXb8BksVT2yJZiRPKX32Pbksp1jek2z/Cwx3NPmTyOqKI9QFJmpdurjsdSa2kWZAZYTB3XyOF9sZqQAcaLMmw7k3jZMXOzvMPmHyN1uY08q4sZstjCbTYHI+VsevaAYBHg6OBx/6Z9BA5uNU7LV+AS9JT+DjsAxYq7R+zGhne7DxYeAlbDNh1mYI/RApmClN91SRm1/XzLrrBEFHJsFnqe6Ix3OnAGx48Zmt8p7hLvymYCXHINDfh6RwEi/kxfwq4OFx7GkSYb6V6kbf1zqTsqOFQQVcIkX3WFuUBJgZWoSCR8aRHa4Ve1HHfE3NLGcbWg0fugTebddsWajabVR/vaGOpODDxqLHA6/Kp8fqyB1Leiy5GyGLDA3ivxnF4CCNP/HkKLpGNBMyuFWHfMP084FP4eV3vZFgnspudWRRlSImaSZBVaw2Q4tWeYu4HnpP72oHb4NScqq2yWOXdiCVFuRKbLlP7GU8x39PHcKKzyMXY8EPPn2O2YyjyLAFHmJ6mAViV26MPiNvxpAT9OormgXgFT01Z/AV9DqMQ7KjpUh6mPTTu8fSBnYJuLzpAvnZKSnDIJDUUJ8O3be8djyfnlUYlMCTPFmiSRhFaWwMNTrdqgqO1dIjesBWRcwmr8rYApJrUJ0m5txqKlC96fuUrDUK/b4yFyKsOScqbYjyZnwWiva/KXkGcZ9uQ3tpgM9bphISTCpCxHtJXywK4w9k+DbLZdvtIbwNdYPwO8Dei1l0I9xz4XmG3ZfR6ybe33Lj8jc/aCWFjo8a4Jb9PBSXNVCrpoIV2OqGQgTOfGuQT7x8p01EW5ZEnivxsgYLhGZJLahK9jI2TU0Z5EnjuBLf3YUDE3mP6CaNL0zEo41bB8CkdG2Z3/PTai1zwS8PMRqK23YCn6dd+Cv6L55kdnAXWnuhivMcTy8pBaj2k99GFKQKxdHxvbIBWNRr7vtDkHUAB2Yze9QPWHPT34Vrc/rS2LV/r6AeM/qmjxJTP3ejt/ufm8DgeHkL//EPui8SLauB038/pfK2ZumBOj/ctDAU40pXeH/YtVswf9W8TAVU4H/k0NVveN9qYhjrVamxoqxI+8HQDKQxMLgMyNIkMx8a38qzClz9Oq5l7ETOpXBclxXYN6847tdQb7mgvogDk5th4+ckeMc6WAhT/Cuzj8xAXRM4g9QN0u3myjJJ79oF7Q2Pjg09HpXLgFspqwGRuOqBAhzCaBxSMe/WgA/yXlDwX5GJO5/f2X9jslTzuiLP6FNE5TbCrgi9/m9r3nAUduB8P8QDPpSNvDrobIehTE7CIb+l8h1rCV7V8DuTsqxhYPvaD+wyHsEWpBWPj/etnJ0ZxDn+43988KE2KJ64LmhKwYhrvsPoCbc5jfRXweAKPY+2fRGVOLaJPHI8b4Q52K2du6gPZ+pamJDtBbLeg6aUcSR0/W6Y9GG6ZdrGCYta2JKc0WZQU9q/bHx2J2NwhsNyv2KE8M7Xe2RtYQCQ4ucaN/u4F3NEV1KCLF0kUk+i/FQkTeR7IolMidGCE0IKWqHwhD47QG2xzFzZQX+I3JZyiA3xEN034xtkdzkHwAE3vTWPWghgwEaq6Isghxr14rihjwMz3/rzgrkb21QaKkvh/j360SSeTMAUPcyDkwE9k8Q86AisHu3Y86tZ/hdqTfpSxnTbU+Q45rufKcDLlhmfBzXrCXmEiWfcX8EqRvBCq7A4WNcEc5thHSRV9oht46IGCzAP+c3SIoi/z4UNVB4125sRn7G64373fOp8ElBkOKi64xxpLnpTo/BCA4ZGYfRvUzqHFBvYjedh1LI8kKvQuRZZE6f3G9g6LB+CsdO2me4j4yjmlLYuuaj7R9CmncUONp1t6aSACZj20ARV9+6hARYkJdHlDMNa4dz9sPIch4kf3QAKdSqoy42TDQV15WOnDa2ZSpSN9HOouwqhNMrV3oub/jgaF5gVGnu/LY5K99HCxMERxSoRPwYiWlgu9Zydvb1k+CoNma9JKVTwtBBb6TsXwT4FWqDbjvaWDlgxKuoxkEVQykeWp5sPBEITD4LDP+v2+rHsGCATPFHwkzLxdO5dxV1FBHVFBgCWATKhGKjMfVvp5XKF5rDjicVWx9ed1SHOi72X1NpvJcqojdqhKwlp6NAHV8dq+TjCwhkqfGJjfgZSLNVvxNxZlxVMWcskwuUgt4wSX72yyf9a974KpFdenZ8z4snvLQO9pWBaHTrfJqH+R+7AgGEXzbsOTwhXm92yDqCckaxbdTwr9VlZENKOYPJbAkrXRaBXvZYNXDnA2xqoqcqnB6hpLqum704OfpZZLBvwG01JcED94xm5UkR9Y9i2RW4jsiipHC5ZFHtIoT1mtiuLOrcdUG/RK5ZkLA6gs0awmqaovfvmHkPJnCMJtPKGP9W5baVothjmmDNNjN0OcZkpPyuolRRCgvbSVUQk7gEeMtpz/GeYVzLd0QTy8Pc+0Wk+ZskxlFnZ3Mjo62wgXq1Zdbqc1PBUjyy+nJlbYoHBlOp6QBX1WmoCfBLTZaKXUgiTv/FTGkqJ0W6Bk7qxh9sTFNDL/M62/xFu63IK3OQ8kgNeok5Mr7TR9jFwMS0SyiCPHCk9BpMoa6vzke3CKLONGwQPlcR1ieIlqXMk8eXGhGHsj0AVmFcrBvgLXnE75OG6UhxkYgFglF1agJPp/9fa0cqrkLkg/WWFxEBSDo51fQQTTYfSa7bZPKTYWq6sOzFEkM27wMzXIKRVBN3/6t/9l0MfzzlT79hb+luUFaqMbYM7Yw4O18Q30+PaHpfdHn2/pg8oW1MHr6ubyd72Poe4Da6DfBw1qPM/JmgFdgWbAXfyAu6yfLOsCjb/Jl1OcwXD/oMDAB7H09Vlw4VLdrvEtIYSidMxWjXQSCJ5h+aRtTKuqP9QRjCXkwUnCKiLKLTgsoL6CJmTgywpOGH3cAl/WhahB7w2OAC790PAxt1CrUR+leeqzpY/SIktQa5s//n3311mKojBF0047wPE8Qkky6NP4Dg76hZWG2w/6w3v7GnX6el+elb+T+lTlMWoYPFB/MQCGpV4bmupLslJuGJMVsu9nPH0dwZqZ1UOZ9zKZpo4+g3uLgKXHGEMDHWdQvM4otdyrymHQzT3dOYrTDJ/cCvkXmUe3Ql9Tt6eUvI8VhgD5t34uD8BW941u++K+Gm8bU6+P/rciYXlKPi0mD+0xmQl8B7MhKUe2YS4/Y79KpK6AlfHvsFP1qLo1hckgbUob/6pEnjZGuaEw6+6uskl3d20GeMF61om/FAmVprijnyqLZFhq+6Xi815h6UeqWlCNKKi/vnFXOwq35pTx9KhF1dR34nBbEwz49A47coOicAfG38tSH5vFxdLBaiQSsW8+vvvaXCvvd/JD/uWPjIPxKFwB2qBSbu8OltiwWqlCjqmImG0bVeHJ33FQrTVA7jTCtuqEaqTF6b3NI8WRbRhoS+fAaWuiWbH8C2AZnmCVdhO4X84C0y6/KqpasBQEp1QFd6KyVhGzmdhafHddFZN8XQ76jf34d3U3+8d/vJNiuqOBsKI/YyNxQ2Mw1+V6uTl4tMVIWzngxrYAeq093O1g2mLM6mQIFgU9/Q4ErDwacVsgfKnjykQe9YMzFcG3jDZpLHizzTXUZ3HaZf2naKoMF7QCTcNQtdLkVscMC+sGV46MbsrjCI3qSzBBsBIshgpC2/nYru5l64N6ujUG1bJKV7ID213wcC7MaqcW078n7dXSgJObJaJQX9EpEGQ2i0SjpeSYJSWHpSjJknRglXxnKU1CR17LSZiAYVmI1SxLeLZy2eRPVNezdLV+1eKedZv7167wWd0yUxU+dS3Ps63YA7osl0wuoF4XnOwZHt0J3TwB29m9KqNHO63hiLIhHSdDZwHLgImErPLdXar/K8E3RLKd5i4e2DSrVajU6CaKkqqju8+42D16JrJW/ZYk/ks6K3tT8kNRd1UAWaqOJDyrnIiuNA74cooHhs1K4Q+rPBXe1QAmcxw9xXQqvAoPbXkQ3ixrR+hPylmWH0wrXVPBG4A81WxLjoalvYIzegWfW9IU37AwlQPYFdjFEXM9GLKAqycjl5GsdPDzEUAwwNHJhQlyloP2dW4jVMzEIRGu8xeUVEVQO7+82jqVy012nA8v3//25ft65fEZSCQHU4RN/FEe6Hzu418ywerXVHEc9z1dEXicgKGxlFHZrWmeUoQR7G/aVwv8abJWKTuN3HNRliM9l1VT/BCTj3Bb5kJMq3YWRlb0WKkq6XlLARQPa3hhJYd6Ha/Wkj8EFCggSkXDvdl83HetVkXWHJWR5WEocEe8HByWDWgfWGarKtmZHgOiIkZUJKQaEVMWI3x9xQ763UYmo8ShLX9JxyC9w7RTe+pjQRijb9N/GOlqMU51Hew25Hz7gapFrcOuVczREGD8irQCnofuwpHLTNMbPzp6qCdGEx2v0V/BQfCuzFIv+UoCNelgSkLFKceVqanH1JfEp+xdHVvHg1ZiUx4Jhq9FmDmyvKSDiazFZ9GFIxAH6oHONhq0RTkpGC48hw61jCfNshjV7QslMDRVTTRLUOlbZP11UUA86xZ/mdQrMvgvRe2PS1aaGFvq0MBn1dK79TIGIKLmghKcej/Qz29xT+dySw6OsciyOB3t7c2BZPOp7UbLykiqg9oDI0PwVKR7eDYgzfbQq0fBu7fZSTcq3/cAUflljy+9hwcoQSnRuVE0CObqLkB24ywt1o8eHR7WpCs8rpMqyCH8c6UpSGaPCpLnKWaTcjywrOl3UC+wppvXi9vgk7Vy0XkSpHwmzOPuPyUDVGme8TxbjE2clVWMs2XHG5s5SyADrJbblsV9d175RewhFSP5stLBGR4up0aTaVB71DgGEA23VJp/TS5KPqlzULVEnVuv5nFKBY8mREClvDd6vTDq4Z+nkkV/5CPo9i7e+YyYBHhkMHxE6zYYXeNS3dwluC2rbYyrI3x7+hJtVHA5kurzDx9fAFIrVTsqxdyxqElZrrDUM/RnQeqFDDP2hD3sN4rMyYOw2JMq/2EjGQT1ujdkgGJtSwGmIMYxzKQQDxPe+9zvHffOvvrehuUpV+N7FBkYC8UuZF38Zg2w5boikgu+tPHYRGy2+PRgk/Cy3vGmAsLa5JtQ/X5Uzw6q58o8z7bUF64TpaKVM4Wnu9SokOWRjfLPTjcqUaDwLawMnPJaEVMF4btU/90XEWJWPJN/+Dqq/FETyrNPZJp0ZNvVvxZAUqZpK7ULpDWhRLJGCoJ2UbNZcrSS/v11aUkONDAaEUreliVakxtKLUBrKfnLge/8X4qs01Q=", "estudio.html": "eNq1PF1v20iS7/4VjDIzJBOK+rIcWzKVTRxPkoWzycaZnQMMI2mSLYkxyeaSlCxZEbC/4F4OuLu3wz7cAfdwwAH3sG/3sPkn+wvuJ1xVd5NsSrLHM3M3QCKyu7q6ur6rmpnjBz7z8mVCtWkehaO9Y/zRQhJPnAbNGqPjKSX+6DiiOdG8KUkzmjuNWT5uHjYAmA/HJKJOYx7Q64SleUPzWJzTGMCuAz+fOj6dBx5t8hcriIM8IGEz80hInQ7iyIM8pKPTLJ/5AdN6L45bYmTvOMuX+DtIGctXzaY7GTzsdDu9zuGw2UxITEN4dzt+dx/ewyCmg4dd0vV7B/CaL/LBQ3pIjyiFt2gGb0fkyCcE3ojnDR723b47bsMbuwIs4yP/wF3vPVq5bNHMgpsgngxclvo0bcLIGnliucxfriKSToJ40B5OaTCZ5oNOu/3t0CXe1SRls9gfzElqIKXm0GMhS+U7UGMOx8CWQWc/WWjZMstp1JwFVpMkSUibYsA6pxNGtR9eWxmJs2ZG02A8ZHOajkN2PZgGvk/j9d7DnCWrhGXARxYPiJuxcJbTIYwCMcliGNJxLp5SSSE8+kGWhGQ5GId0MZyQZHAIgyQMJnEzgJ2zgQcCo+nwphnEPl0M+kOEbF6nAIp/rffsjE5WNTRbp+YiMYeCb4MOHpSFga+JWRRQMdlMiR/MMkFbQnwf+d1LBGndZCG209xZnrPYst08Xim75SmwJyEpkFzjMki53L29sdMTZSN41jqwi5BIEE+B0YBplmaAKmEB58T1FBjThG08OohZxQNJlM1qJD3set1xj2wJvbZmAOwjbkj9FQO0Qb4c2L1+sa9Px2QW5rggWSF/Bx149v1K1CkNSR7MKR8ebTLl/iLYQaPva3ZE41kp4JjFdLitZEKj2kLZOrs0/1fqwAE8R0EsnMWgc4RzEVk0pbUdtOfTyiLILGeceJslNN44gRsy7wpmcVAjlvgVcqjDDOVeaMg5XeRNbhYDNKNtG+YAPvVYSjhnBJ8k8YeoV0hx/YB4JoVPfEmppXeoYEn8YIpHrh1BDO3QwGJRFpEwXG1ZB/CN5AMuRvAkdJHcokfsqhSTigOHkeLmdSGPNqBxGZAU7fBJYmLAbY27pT7yGK13zNJI2DHoNP07owkzpuJ9ao6GuyI/SKnH0QNBsyje5b0Kx4YaI6R6dDC/BgpJHETZassJomTqfm74eZblwXjZlDFMYi5QFAr0q+1OcVWFmnQ37OAnXdQGUfYuuiDU7RQkHxcDD8fjMaAK4jHbIUIRTSBocZs/QBq3lQo1AoImFZIupLg/xHMXptux+7BLlpMJdYk/oSvE10dpSQC0ZkVyT9pcciGd0Nhf3eaK+IZSzZ5sOIv9/qazUM1QZg6e94t9VWnuG+ev+dCSGeVZND+Y71bFHRot0w1Aq7XXBYJgVbisins7/E5PoSWIuSykU3wIXmAXT4MYsjtwSjXqdlC120i2FaM4fG+955F4TrJN7/ybiPoB0YxK7Af7cBBztafxNKfgDOrKYeFEDsvU5hAzBW0rVVCN6Kgunx5fIZQdkQqtEPqzewM1JshzKvg6FT7NTWvhE8elZ5Qb9NoCOmcszFZ1J1UqanMh45pWdznFofo7tE6easO41LN1+u3aoeoqi4ulodXgNw+63jtuyZz8OPPSIMk1rBycRhBh5h+RpDFayZesMVg18mlKaWPQmOZ5kg1aLc+P7c+ZT8NgntoxzVtxErU40G/aduegbbdb7iwIfTFmR8yfhRRWNCyBqgVMYHHW+nko6YJEkGVnrc9Z1Gqs13AITjycoiWqG0zt4Q0sUwt8pwHcg8JE0/iAF5IscxqgZA0+iXVOxqcBQAhH80lOmnNYSBeNYgGLG6MPoDuzlBy3BNxoEx6SWGDU6F1Ab0hWQu1ELSIuAtOMZbdihM2XjdGz1AvC8PZ9pwHAhDlRtzxuwWm3Tg2M2Dnh+4IbIVLVKDeQs2CFjdHf/vlP2tnsRvvbP/53RYiCAvOUXXwMsfDkxWCjqAp3HwQBF2CQAUsBUj5pBjgz8w5ewir4AUfpUVj2rHi8dQs/JREkex5pjF7A49c/4/Ot0GD+MUC+wx/NSKH0zUiq5UIPKrIET+/krEeiaDdj/+df/v5ftZOvf4bgQH42dz2n0bbajdH3WD7R3ecAmM4hhzqFAij0ya1gRxzqjPhMC27+aN/BdgBuKtAQp+xbsfb6VuewMeq19m8FaVuHbdTzNHBvp24MxVtjdEJUC/xpxkNWvJvvpwt0bCDOO3le4BACqO1X/BR+BsMwp29CYuAJuDsasr/96d824SrH3tC4/wVi71Oo1dKxfi37ke2PKgXcTj13pKf1mDPkzZIpiPN60NbaWgczIon58LB0HNVxeYjZHsfouT0qIlHlh3GMR8RbPFLpoHl43VB9HE9ZXnPOzyA+NmGQbNs+t78c3MMJGEmqAGwhvYayBMydhV//Y8LiHY6Zy++KhmhLf5zRkOZsN8wUyAPT/oe/aCckqUWNyjkXP7XgK6IkHFgEXu2RRjLtw6v3p6faOGWRpvNoqA+L+dXb1A3yExBlCoxaq0BFfPXkZKsGCnFWwfLy7MP3Z6DAUIjsQhHyqaxVgdWXv3j/7OTtPdYrcHUE7xmLTuN5kLIYTC3fiYRW81lrY4HAtgdHzXLtGydzRj7zZnwG5JQuz0FWXs5SIzOHEur712en584KvPngQhfW+vGauvYkdHWrGEAdBZOoD4oAENwUE5cWD/2AJuGxXw7KII/DGOXlKMZ0GCIiqMvBaYA0QJFPPhKI5GJ0XVD6u9cnp85qk8aB/vLsuQYhm6ZMg0HNgIjmMe2x9tt3py/N3WcQi+SIZsAyS3vJfJZb2geepX3OlJXgeOyxu4BV3z//O+wEa7RQe834IQ7ypaX9EKeUhJb2JoDEjMHiPU0ut90QPAQsfo6/FIK5GDFv56QgLwuUjSpgyCMwdDObuZ8B8u3z3wJUzkM4U7b9CDqV0gw8aM+3szwE0PMPZ5oc/fpfsdZ7AUVKpJARMr/zsd+WJJy9fVGwtd/+tgbV/djpK1DRbFlAdvrfchoUDRjoIhXUQMW16Qx0AHBVqjDQ35CEQOzU+BjMqToBRIM317C1QpCEum7gWhjQcECDY0Gc8PjRDORdSn14TfGA88Cn7CNkH0FMUjtK9mHpH3BM6ABfVIJNwJp6kGSrYPD+1//kEMD9jwnmQMRO4gkAvI4gkMV4AD6piUl9XdphysVOUyem18KH2T9S9+XZezlurEiM9wcByQZ5OqMWSoimcwqKfA3R6/lsPIZaHqfWYLYFOigR83fBgobvsW1nvCH51IbzGeJqopqxuqY5VBedQ6wzgjim6Y/y+gIeX3ErVdGzWZ7MwE9CMD3HjrEjSD9///J5NVghBqdOQZIJECwhn52cnn8fhFHgfajmFEp4kIVhm8a8g+zgCYd7pcfCAsaGVbDgZAoFlFEu9cHphRSBSjeWQR1LFQ6f47thDvm4XWUJCgg/hdFeiOuXElMSpTRSwN69eX/65iVgSQl6zoIIbDUhOLhjqIgcvspGdy12xvUbztkwLajc9k1bmDstNgS1VLeDAJFgV3BOT0gEmxqQOVbSainSQnQdq9tul7R7echR1UKcARtYO3kH4JL5L6CaRMlxEeAwFtrvIY/I6a6x84SCvDp2t9h4kgYqa1/C6ysaJqDcnba1Dynyokt7HfHQ7e53vUIyENQMXFweYRwypprKG5pNDYU9UITQl5RFNE+XRrcNxzctRexSqbBuIqGhXkqsTd4qZqmN+RGmlvbCaXKzefe61S3moG4C1gs84ugVoRykpHRKo0Ah9BW8ZskUmHyGwgHFGvP/4Mg9+G9/37IP1FPj8kp9Z7GC6kXRFybhBq4uYJjFtkeyvEYijAl7siOSoIGjoQN79g8t/EvdF2DLbdMgunvbw0NCYNuOigAWAYI9iEqaNwNRzcLQisDdyEfCUWTOag1BnnlXqsXhO5gB5NjK6HO26MHgK9CnQwvTSqE5Ahvm0YiKzEkgdPscDlYeQGRUfLxKyozCNn3MBPikknHBLB9HDr2gHgS19B0ogaH/0u5LGLhZi6NsTcJ83NLNoSCL76BszGGQdeNZzJmk8U6DcYW9QU2QfOasZKNgcFFJ3broWvtW79KqJALaZHXstoXyv7SKnoFc1PV9q2fDsr4FCoDL3LF/CMvsvmUf4VKxDFP/snEg1pL2ARgVQFw092Fjvvig/aSNewJGMCT+R6wv+wgKsT3Y46Jn9a1mB9eOx/3+kycWytYGojt9a0zCDNbyroKyzt63LtqwrC9WCdOBdQd8X7FofXF1OQSiuQ2g+0YWv6IL4+yifSlMA68RYigdl87ZReeSDxWlJLcJ27bPLrqXph3NwjxIwuW5R0KSGq8gSAbRJtbepRhWse5zEnC0hrjZffTKwj/N3iNAhuZdW9a/lDakJPAw/OTyqQghA9T3bZDXCooDvnMwNh7gOnO1Y4/H9gH2NEsNc8HWDHMFhmrPEh/8Iji8NFj8yFIIqChEvA5YgNe4oqdRki9L23G3LBT3RjxQPM4hSFGDOaMVEMPsIEMvbRYv51cYpHwxBnZ2xTPYWBIAiBiwOYLkgj7HkIwpDhI5dGE4WQIKtxo2MQMIl4LqfZiLKvrNNS9dKWgGbDORQeEncJdg994E+TOLgZmGa67XmEtB9AbOsoXC5zFYAjXIjQO2QUOn0+U2LbgvOeo5iGlC8xPecleC2h94UQYsNq2sAOJZ2i6Q4StH5HpkYWT20oIcQIhGuDx1cmEhQGbfmI869lGLz+QkxoTAHrP5oyL4QXZrQgQskRCH3JST2DWjDg3VAQ5KNgzAsxePOQwk3wYxBbzHMoOaj3zLs5fVrBy5eVzCbMKLI0HSAd4JuCGkh/d7sGtMSer4LUh8+OsY3x71xYvQsXcp+yxCmZCpIXKdUgFVC8FyKgMTwXiWO21r7rSHd6n5d989YPYMEvQX4Dlt7JWAeYkkyKnUazh/7ExsksPuLihjVrHKA73LhzlO8ybWU/krJlq9wR3LWj1UwFJO91AWgP3G0Plljm7aInn88ObM+fTNivObJ8VGDkkpO2P4/dI5bB1PDHOtwcPXP8eTWcgy7a9/0b5ZzXcAzb/+ewoBgGbHbjr6ZgU6B0DfQzbgG12Y/vpPGg4udw3e1AajT4/Rmg2k1OZlYvb0k9hXGVqLCjL7NNB1MH+SLWNPK0VJF1CBZcbYXIEEpFBSh1yTAPJKmntTY2ytQDxT5g/0V6fPXuiVOUPBc7X2CAKBQOUgjzrr9d7mRtmUXRt4aaJEbo/EfubwdsoFTl0OUaPGQUh5JjOEGtaQKa7GxgLcBL3imY09JUi3ueLw46GbUnK1ls4ex0xBUSFPTDBAnrxxasu+qaPj3aZeqcdEHl1mI/jzDM9hIEKwaa7WFKSQk9BUsGJ9ciK/d/u03cnVaqrTabcfUY6b+q0C2frbT0JPYQOwJXMlwhqUSGxO+cgaZqpcbwNAmRhuJIRrEYTAePiK4d0BiW1lymxXhs+gcJtB1hWdwA5Q2HCxD3lwAXQoTMdxdGxZ6V++FK/TQDd5pOClxkbFcg5u1oe8qqpERFcaErEj1/MOLGDeZBrTLBvY/b6FnxyG4q3dF5yrbS17aDqe6N47Ak9AkrzKlY0FZdMna4hlYqcqs0fBDDfDVhRzLxMFsb3k7C5d0rLpRLGsm6pBcJ9Y0NVHAGlVIgyrEmGIasc74DXvpOtSecAXYoOGg9shjSc5SFViKo//rAB5gxPlKQoz4C1/50J/T4EkbD6diD4QPrEUamJ4OCfhzCfyKecPz8Es+cMbEvO+1CsolPnAO5bRjx/0S7FJjcSMpblhEMs1nZHBNxbO/S3YuY3flJqPj47Mb4+Omhuzbm1W0l+5DA9dhrqTuZJMvPD4ykuHc8X2wiB5xmcwWnIsBR9cp2yteOBecir7AIYuevI6Zkmq7QvMYDDgXTxq6B/h7BqHYjHsA6WdAcdE12MIULMmTrV148pcjQMXlBcSuXxaPA3UI0JKL7GKo8nojU0Pjg3dK7jNc3AQ0mJvcwZqQlqskO00JBhdDNBbfMnoPJAw6iS/YzmD8FL4KJ0zDM2H5j9CEmAgMGYDNWC8dCBBnAnwMhyLW6MtB75p7pWz4Z11/anOvzHRBzpekelb2CoD2sT0VD92y1v2EcbqE+ITjTslzdFmsYirNs68YKCTcCwtojdwktjSQqLxzisO+gyswNYHXJyb5OEmotkLsQL8NE0TvBka6Y85xBkn1DAHwr5FzgzVMq+CQXhnogcpM7Lh7YGuPP7u641nYQhL+UcNWqHbkPCmpwSiu+uMXEVCOZtMQiFOy7XxlhUEas/hVDy+13PF2ilkgpFRKno3b0iCTvMXpY6R8yxNyRLg+K9RuXnzafUMFjGo3qSLjL77LuJ2AtiRFp5KiBHT5AOYnIsBS3+oPy6K3AkvcotUzpTRQGY+F1An4+JLbCgZxkVseZdg7J+O+dVvUFzZKtev36y8NV5jBpAGxmtxq/fJtD8zyPd5siYbRuitRRhQ+IpuIS6YAeWH9BDx5RATIFJkPyKdQAzffScfHjgOMeUzlAI+fTuDIr9nDomNbfTcMPno6xgHbb4RFxIng/yUDqmfK91Xh1QX6jjxlv8EVivVh/QeLObl8wMMXvKwghnXmXNxeYtW1VXpGqqFWTY1mAmShJeC2GtndG2LT8sL53Vt3lrNPyiU9cuXzQ2qFJQr3lapb9xHhwcX1fNlxdHIGUV8F+4UHGCHTMzX5XalflxvtoerssuqzzwnWeBt52CH7qE7HlvldiI7Uj7BFwNK89jyaZJPf0yDnA5EUWCW0fW6ziaRUl7bouP+licfXUgxMcG6Ritbl71T7u8cXTb8dOXuY1sVi+/jN7WwCMWYzlPwkyyB4jchEx5Dq06o40oVkEFfjjPH3x2tIHSL8HY3SQo1C2e02BUmOSaT2zEz1d2QIXIaXU+5EYyfzuEBgfDaxdD5CXULs43/I3LMO3l9Ib58uryN10jIqpRfFTXCzVC2vs8+3k/sU7+NEZkOhsaU5XcmJgFmlQVpHoZp/HYIaoif3ZWSoLmzPenZC0s0EBdQD7x6ZLefYGOnSMA3e0Q5gOf2Ev7cPAbgfl80ZtQmT77Rq6ncgKDigliQ7SonsyEpCCCLtYAbGKx+N4tcrBxlS86iPyWGOzMFVRa87FeSBFSjShJ3Se1B/X2n/NQoUgc312IfL0839ilSqD0l+6zTIcw8dh7cIz8d7oRR6UK/rIQsSZfMke/YeCuL3rHxFszmxntlHFCbAtXdn+wzy4tHXgZWgGXQsbkr/wBOV9rSLhj8slz8S7+uenWm9CWEft/ZffiV8XAV2UpIEmElsmVQcviNJo8msgN+3w7K8P+b6ILCzrB+ABk3keZCb6bbhlNeVIsHgx/LAkdSeCFyaxlLdMz6phDUnR3X3aBOGKN/eH9m6AF+s9HC7zdwhc+uY6wzHN0TX6nxLztggpNloJZvxyT8iuaGiqC0Qj9H+LW9s/uq/u7e9H2/ztj8/KNsfJwxBkl6ZXY+FO9YKaJff0HDnBiyiIZaQa2l/W1nexv3ce89g/dB+Uagdthk5fezos1oqI1WPQ0mH3nj9nPGS1/+C7wsW6yVR4fSsrzZfesic+w5CaEeFc1gknrTYM6yL19WMuvaaKVufTNm/boP2Sz18yX1W6X6t0mXJuwqvgvYRZU8yRVdZgZ+umZugAtugTZEQQbuBQIR1lzwK0ou2XEW5ilgy/a2KXrGApfkyS+Nb1XXQ7SulfB2aWeQVRtAQ61HrVxARA6asfwwWL8fF7Za3j/DrMeqteJL1fFA5Bfjy8eftGP+7wShCh0XmYENh07AQkz8ZxJ88hP4JrU/Rbgf3aCdk4n0PpDIfx2tal043rm/wP57bgzgp84pqv47AjE+M8CciUyjcJTzGsuIufN7tHJDn+vmly96jjcASLK8iLhNhnNHlQwHBsUIYt+4ckZi7dVta2Eb3guXAVnuL/IGTDM7urkzkRAJv8xO5ddTeUE+zUmC3MMribxgtMv1q/pUfKtFSXMY2GgOiR7ZujQxcVmDjRwTM6XyX8oct/g/kTluif9JwP8C9OpCCA=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()